# 02 — Préparation des données pour la modélisation

Ce notebook autonome reprend la préparation AI corrigée et auditée. Il produit
les données **avant imputation**, leur provenance et les plis. Le code de parsing,
de nettoyage et de preprocessing se trouve ici ; aucun import de `src/` n'est requis.

Les expériences finales utiliseront les mêmes partitions et ajusteront les
transformations dans chaque entraînement externe **et interne**. Les cibles ne
sont jamais imputées. Voir `specs/spec.md` et `reports/08_notebook_migration.md`.


In [ ]:
import hashlib
import re
import warnings

import numpy as np
import pandas as pd
from sklearn.base import BaseEstimator, TransformerMixin
from sklearn.exceptions import ConvergenceWarning
from sklearn.experimental import enable_iterative_imputer  # noqa: F401
from sklearn.impute import IterativeImputer, SimpleImputer
from sklearn.linear_model import BayesianRidge

SEED = 42

# --------------------------------------------------------------------------- schéma brut
RAW_COLS = ['C', 'Si', 'Mn', 'S', 'P', 'Ni', 'Cr', 'Mo', 'V', 'Cu', 'Co', 'W',
            'O_ppm', 'Ti_ppm', 'N_ppm', 'Al_ppm', 'B_ppm', 'Nb_ppm', 'Sn_ppm', 'As_ppm', 'Sb_ppm',
            'current_A', 'voltage_V', 'ac_dc', 'polarity', 'heat_input_kJmm', 'interpass_C',
            'weld_type', 'pwht_temp_C', 'pwht_time_h',
            'yield_strength', 'uts', 'elongation', 'reduction_area', 'charpy_temp_C', 'charpy_J',
            'hardness', 'fatt50', 'prim_ferrite', 'ferrite_2nd_phase', 'acicular_ferrite',
            'martensite', 'ferrite_carbide', 'weld_id']
CAT_RAW = ['ac_dc', 'polarity', 'weld_type', 'weld_id']
NUM_RAW = [c for c in RAW_COLS if c not in CAT_RAW]
NONNEG = [c for c in NUM_RAW if c not in ('charpy_temp_C', 'fatt50')]   # températures d'essai peuvent être < 0
MICRO = ['prim_ferrite', 'ferrite_2nd_phase', 'acicular_ferrite', 'martensite', 'ferrite_carbide']

TENSILE = ['yield_strength', 'uts', 'elongation']        # tâches principales
COMPLEMENTARY = ['reduction_area']
ALL_TARGETS = TENSILE + COMPLEMENTARY + ['charpy_J']

# Jeux d'entrées (spec : A0 = cœur 11, candidat principal 13 = cœur + O/N, 16 = + Ni/Cr/Mo)
CORE_11 = ['C', 'Si', 'Mn', 'S', 'P', 'current_A', 'voltage_V', 'heat_input_kJmm',
           'interpass_C', 'pwht_temp_C', 'pwht_time_h']
FEATURE_SETS = {
    'A0_11': CORE_11,
    'A1_13': CORE_11 + ['O_ppm', 'N_ppm'],
    'A1b_16': CORE_11 + ['O_ppm', 'N_ppm', 'Ni', 'Cr', 'Mo'],
}
PRIMARY_SET = 'A1_13'
PROCESS_COL = 'weld_process'            # synonymes documentés seulement

# Synonymes documentés dans welddb.info (« ShMA = MMA », « SA = SMA »). Rien d'autre.
PROCESS_SYNONYMS = {'ShMA': 'MMA', 'SMA': 'SA'}


from pathlib import Path
from datetime import datetime, timezone
import json
import platform
import sklearn
import matplotlib.pyplot as plt
from IPython.display import display


## 1. Lecture, unités et provenance

`N` signifie non rapporté. Les valeurs exactes et les zéros sont conservés.
`<L` devient L/2 et un intervalle son milieu : ces approximations gardent leurs
bornes et leur statut. Les unités suspectes sont mises à NaN sans conversion.
Les annotations d'azote total/résiduel restent hors de N principal ; la dureté
garde la valeur et le protocole séparément.


In [ ]:
def sha256(path) -> str:
    h = hashlib.sha256()
    with open(path, 'rb') as f:
        for b in iter(lambda: f.read(1 << 16), b''):
            h.update(b)
    return h.hexdigest()

def read_raw(path) -> pd.DataFrame:
    with open(path) as stream:
        widths = [len(line.split()) for line in stream]
    if not widths or any(n != 44 for n in widths):
        raise ValueError('Chaque ligne brute doit avoir exactement 44 champs.')
    raw = pd.read_csv(path, sep=r'\s+', header=None, names=RAW_COLS, dtype=str,
                      keep_default_na=False)
    assert raw.shape[1] == 44 and not raw.isna().any().any()
    raw.insert(0, 'raw_row_id', np.arange(len(raw)))          # indice brut 0-based, stable
    return raw

_NUM = r'\d+(?:\.\d+)?'

def classify_token(col: str, tok: str) -> dict:
    """Statut lexical d'une cellule numérique brute. Aucune réinterprétation sémantique."""
    out = dict(status='exact', value=np.nan, lower=np.nan, upper=np.nan,
               censor_op='', annotation='')
    if tok == 'N':
        out['status'] = 'missing_N'
        return out
    if re.fullmatch(_NUM, tok) or re.fullmatch('-' + _NUM, tok):
        v = float(tok)
        out.update(value=v, lower=v, upper=v)
        return out
    m = re.fullmatch('<(' + _NUM + ')', tok)
    if m:
        out.update(status='censored_left', censor_op='<', lower=0.0, upper=float(m.group(1)))
        return out
    m = re.fullmatch('(' + _NUM + ')-(' + _NUM + ')', tok)
    if m:
        out.update(status='interval', lower=float(m.group(1)), upper=float(m.group(2)))
        return out
    if col == 'N_ppm':                       # ex. 67tot33res, 66totndres
        m = re.fullmatch(r'(\d+)tot(\d+|nd)res', tok)
        if m:
            out.update(status='annotated', annotation=f'tot={m.group(1)};res={m.group(2)}')
            return out
    if col == 'hardness':                    # ex. 158(Hv30), 459Hv10
        m = re.fullmatch(r'(\d+)\(?(Hv\d+)\)?', tok)
        if m:
            out.update(status='annotated', annotation=f'value={m.group(1)};protocol={m.group(2)}')
            return out
    out['status'] = 'unparsed'
    return out

def unit_review_flag(col: str, tok: str, info: dict) -> str:
    """Heuristiques NON résolues (spec 2026-10-05) : la cellule devient indisponible, rien n'est converti."""
    ppm_review = ('Ti_ppm', 'Al_ppm', 'Sn_ppm', 'As_ppm', 'Sb_ppm')
    if col in ppm_review and info['status'] == 'exact' and 0 < info['value'] < 1:
        return 'positive_below_1ppm'
    if col in ('Ti_ppm', 'Al_ppm') and tok == '<0.01':
        return 'censor_0.01_in_ppm_column'
    if col == 'V' and tok == '<5':
        return 'censor_5_in_wt%_column'
    return ''

def build_provenance(raw: pd.DataFrame) -> pd.DataFrame:
    rows = []
    for col in NUM_RAW:
        for rid, tok in zip(raw.raw_row_id, raw[col]):
            info = classify_token(col, tok)
            rows.append(dict(raw_row_id=rid, column=col, raw_token=tok, **info,
                             review_flag=unit_review_flag(col, tok, info)))
    prov = pd.DataFrame(rows)
    assert (prov.status != 'unparsed').all(), prov[prov.status == 'unparsed'].head()
    return prov

def primary_value(prov: pd.DataFrame) -> pd.Series:
    """Représentation numérique principale (déterministe, sans apprentissage) :
    exact -> valeur ; censuré '<L' non signalé et >=0 -> L/2 (approximation) ;
    intervalle -> milieu (approximation) ; annoté, signalé ou N -> indisponible (NaN)."""
    v = prov['value'].copy()
    cens = (prov.status == 'censored_left') & (prov.review_flag == '')
    v[cens] = prov.loc[cens, 'upper'] / 2
    itv = prov.status == 'interval'
    v[itv] = (prov.loc[itv, 'lower'] + prov.loc[itv, 'upper']) / 2
    v[prov.review_flag != ''] = np.nan
    v[prov.status == 'annotated'] = np.nan
    return v


## 2. Doublons et observations apparentées

Déduplication sur les **43 champs bruts hors identifiant**, premier conservé,
autres identifiants enregistrés comme alias. Les séries sont dérivées des
identifiants ; les doublons relient les séries concernées. Ces groupes sont une
hypothèse de dépendance, pas des publications confirmées.


In [ ]:
def find_duplicates(raw: pd.DataFrame) -> pd.DataFrame:
    """Doublons exacts sur les 43 champs bruts hors ID ; le premier est conservé."""
    key = raw[[c for c in RAW_COLS if c != 'weld_id']].agg('\x1f'.join, axis=1)
    first = key.map(raw.groupby(key).raw_row_id.min())
    al = raw.loc[first != raw.raw_row_id, ['raw_row_id', 'weld_id']].copy()
    al['kept_raw_row_id'] = first[al.index].values
    al['kept_weld_id'] = raw.set_index('raw_row_id').loc[al.kept_raw_row_id, 'weld_id'].values
    return al.reset_index(drop=True)

SERIES_FAMILIES = [
    (r'RR82011', 'RR82011'),                 # p9-RR82011 ... p36-RR82011 : un rapport
    (r'^EPRI-TR-101394', 'EPRI-TR-101394'),
    (r'^EvansLetter', 'EvansLetter'),        # EvansLetter, EvansLetterC, EvansLetterCo
]
# Variantes d'orthographe (mêmes auteurs + année, AUCUNE composition partagée constatée).
# Désactivé par défaut pour rester aligné sur la règle du projet (47 groupes) ; sensibilité possible.
NAME_VARIANTS = [
    (r'^Kocak.*TiN-1994', 'Kocak-TiN-1994'), # KocakPEv / KocakPREv
    (r'^Gar?&K-1975', 'Gar&K-1975'),         # Ga&K / Gar&K
]
MERGE_NAME_VARIANTS = False

def series_of(weld_id: str) -> tuple[str, str]:
    fams = SERIES_FAMILIES + (NAME_VARIANTS if MERGE_NAME_VARIANTS else [])
    for pat, fam in fams:
        if re.search(pat, weld_id):
            return fam, 'explicit_family'
    m = re.match(r'^(.*?(?:19|20)\d\d)', weld_id)
    if m:
        return m.group(1), 'prefix_year'
    if '-' in weld_id:
        return weld_id.split('-')[0], 'undated_prefix'
    return weld_id, 'exact_id_fallback'

def build_groups(raw: pd.DataFrame, aliases: pd.DataFrame) -> pd.DataFrame:
    s = raw.weld_id.map(series_of)
    out = pd.DataFrame({'raw_row_id': raw.raw_row_id, 'weld_id': raw.weld_id,
                        'series': s.str[0], 'series_rule': s.str[1]})
    # union-find : un doublon relie la série de l'alias à celle de l'enregistrement conservé
    parent = {x: x for x in out.series.unique()}
    def find(x):
        while parent[x] != x:
            parent[x] = parent[parent[x]]
            x = parent[x]
        return x
    ser = out.set_index('raw_row_id').series
    for a, k in zip(aliases.raw_row_id, aliases.kept_raw_row_id):
        ra, rk = find(ser[a]), find(ser[k])
        if ra != rk:
            parent[max(ra, rk)] = min(ra, rk)
    out['group'] = out.series.map(find)
    return out

def build_master(path):
    raw = read_raw(path)
    prov = build_provenance(raw)
    prov['primary_value'] = primary_value(prov)
    aliases = find_duplicates(raw)
    groups = build_groups(raw, aliases)

    num = prov.pivot(index='raw_row_id', columns='column', values='primary_value')[NUM_RAW]
    m = raw[['raw_row_id', 'weld_id']].set_index('raw_row_id').join(num)
    m = m.join(groups.set_index('raw_row_id')[['series', 'series_rule', 'group']])

    # catégories : codes d'origine conservés ; synonymes documentés seulement
    m['weld_type_raw'] = raw.set_index('raw_row_id').weld_type
    m[PROCESS_COL] = m.weld_type_raw.replace(PROCESS_SYNONYMS)
    m['ac_dc_raw'] = raw.set_index('raw_row_id').ac_dc.replace('N', 'missing')
    m['polarity_raw'] = raw.set_index('raw_row_id').polarity.replace('N', 'missing')
    m['flag_ac_with_polarity'] = (m.ac_dc_raw == 'AC') & m.polarity_raw.isin(['+', '-'])
    m['flag_polarity_without_current_type'] = (m.ac_dc_raw == 'missing') & m.polarity_raw.isin(['+', '-'])

    # annotations conservées hors matrice principale
    pv = prov.set_index(['raw_row_id', 'column'])
    m['N_annotation'] = pv.xs('N_ppm', level='column').annotation
    hd = pv.xs('hardness', level='column')
    m['hardness_value'] = np.where(hd.status == 'exact', hd.value,
                                   hd.annotation.str.extract(r'value=(\d+)')[0].astype(float))
    m['hardness_protocol'] = np.where(hd.status == 'exact', 'unspecified',
                                      hd.annotation.str.extract(r'protocol=(Hv\d+)')[0].fillna('missing'))
    m.loc[hd.status == 'missing_N', 'hardness_protocol'] = 'missing'
    m = m.drop(columns='hardness')
    m['micro_sum'] = m[MICRO].sum(axis=1, min_count=5)
    m['flag_micro_sum_not_100'] = m.micro_sum.notna() & (m.micro_sum.sub(100).abs() > 0.5)
    m['flag_pwht_missing'] = m.pwht_temp_C.isna() | m.pwht_time_h.isna()

    # déduplication
    m['is_duplicate_excess'] = m.index.isin(aliases.raw_row_id)
    alt = aliases.groupby('kept_raw_row_id').weld_id.agg(';'.join)
    m['alternate_ids'] = m.index.map(alt).fillna('')

    # éligibilité
    for f, cols in FEATURE_SETS.items():
        m[f'n_unavail_{f}'] = m[cols].isna().sum(axis=1)
    m['in_strict13'] = ~m.is_duplicate_excess & (m[f'n_unavail_{PRIMARY_SET}'] <= 2)
    for t in ALL_TARGETS:
        m[f'has_{t}'] = m[t].notna()
    m['is_unlabeled_all'] = ~m[[f'has_{t}' for t in ALL_TARGETS]].any(axis=1)
    return raw, prov, aliases, groups, m.reset_index()


In [ ]:
PROJECT_ROOT = next((p for p in [Path.cwd(), *Path.cwd().parents]
                     if (p / 'sujet.md').is_file() and (p / 'notebooks').is_dir()), None)
if PROJECT_ROOT is None:
    raise FileNotFoundError('Ouvrir ce notebook depuis le dépôt ou notebooks/.')
RAW = PROJECT_ROOT / 'data/raw/welddb.data'
OUT = PROJECT_ROOT / 'data/processed/modeling'
OUT.mkdir(parents=True, exist_ok=True)
if not RAW.is_file():
    raise FileNotFoundError('Lancer python scripts/download_data.py depuis le dépôt.')
raw, prov, aliases, groups, master = build_master(RAW)
dedup = master.loc[~master.is_duplicate_excess].copy()
print(f'Brut : {len(master)} lignes ; dédupliqué : {len(dedup)} ; groupes : {dedup.group.nunique()}')
display(aliases)
rev = prov.loc[prov.review_flag != ''].merge(raw[['raw_row_id', 'weld_id']], on='raw_row_id')
rev['series'] = rev.weld_id.map(lambda s: series_of(s)[0])
display(rev.groupby(['column', 'review_flag']).size().rename('cellules').to_frame())
assert len(prov) == len(raw) * len(NUM_RAW)
assert not raw.loc[~raw.raw_row_id.isin(aliases.raw_row_id), RAW_COLS[:-1]].duplicated().any()
assert prov.loc[prov.review_flag != '', 'primary_value'].isna().all()


## 3. Cohortes et schémas par cible

Candidat principal : **13 variables numériques + procédé**, avec au plus deux
entrées indisponibles. Le seuil est un choix d'ingénierie ; il ne garantit pas
l'exactitude des imputations. Les schémas 11/16 servent aux comparaisons futures.
Les variables exclues ne sont pas déclarées sans effet physique.

Une cible connue définit les lignes supervisées ; le pool sans cette cible est
conservé. Une propriété mécanique différente n'entre pas automatiquement dans X.
Les lignes PWHT manquant restent dans la table maître et la cohorte si éligibles.


In [ ]:
def task_schema(task: str, feature_set: str = PRIMARY_SET) -> dict:
    """Entrées autorisées par tâche. La température Charpy n'est une entrée que pour E(T)."""
    num = list(FEATURE_SETS[feature_set])
    if task in TENSILE + COMPLEMENTARY:
        return dict(target=task, num=num, cat=PROCESS_COL, status='adopted')
    if task == 'charpy_E_of_T':
        return dict(target='charpy_J', num=num + ['charpy_temp_C'], cat=PROCESS_COL,
                    status='conditional — protocol classification pending')
    if task in ('T100J', 'T28J'):
        return dict(target='charpy_temp_C', num=num, cat=PROCESS_COL,
                    status='conditional — protocol classification pending')
    raise ValueError(task)

def charpy_protocol_table(m: pd.DataFrame) -> pd.DataFrame:
    """Indices par série — candidats seulement, aucune affectation de protocole adoptée."""
    c = m[~m.is_duplicate_excess & m.charpy_J.notna()]
    def agg(g):
        e = g.charpy_J
        return pd.Series(dict(
            n=len(g), n_distinct_energy=e.nunique(), n_distinct_temp=g.charpy_temp_C.nunique(),
            share_100J=(e == 100).mean().round(3), share_28J=(e == 28).mean().round(3),
            other_energies=int((~e.isin([100, 28])).sum()),
        ))
    t = c.groupby('group').apply(agg, include_groups=False)
    # Signature de protocole par série (pas par valeur d'énergie isolée) :
    # énergie (quasi) constante sur de nombreuses températures = critère à énergie fixée probable.
    r = t.n_distinct_energy / t.n
    t['signature'] = np.select(
        [(r <= 0.1) & (t.n_distinct_temp > t.n_distinct_energy), r >= 0.5],
        ['fixed_energy_varying_T', 'varying_E'], 'mixed_unresolved')
    t['candidate_task'] = t.signature.map({'fixed_energy_varying_T': 'T(E_fixed) — ex. T100J/T28J',
                                           'varying_E': 'E(T)', 'mixed_unresolved': 'none (review)'})
    t['status'] = 'hypothesis — verify against the source publication'
    return t.sort_values('n', ascending=False)


In [ ]:
cohort = dedup.loc[dedup.in_strict13]
eligibility = pd.DataFrame({
    'labels_apres_dedup': dedup[ALL_TARGETS].notna().sum(),
    'labels_cohorte13': cohort[ALL_TARGETS].notna().sum(),
    'sans_cible_cohorte13': cohort[ALL_TARGETS].isna().sum(),
})
print(f'Cohorte : {len(cohort)} lignes / {cohort.group.nunique()} groupes')
print('Groupes exclus :', sorted(set(dedup.group) - set(cohort.group)))
print('Sans aucune cible :', int(dedup.is_unlabeled_all.sum()))
display(eligibility)
display(pd.DataFrame({name: task_schema('yield_strength', name)
                      for name in FEATURE_SETS}).T)
display(dedup.loc[dedup.flag_pwht_missing, ['raw_row_id', 'weld_id', 'in_strict13']])
cp = charpy_protocol_table(master)
display(cp.groupby('signature').n.agg(['count', 'sum']))


**Charpy reste conditionnel.** Les signatures par série aident à vérifier les
sources ; elles n'attribuent pas un protocole validé. E(T) utilise la température
d'essai comme entrée ; T100J/T28J la prédisent. Aucune ligne n'est classée seulement
parce que son énergie vaut 100 J ou 28 J. Les entraînements commencent par la traction.


## 4. Partitions reproductibles

Cinq plis externes par groupe et trois plis internes dans chaque train externe,
sans mélange ; graine des estimateurs : 42. Les plis communs sont définis avant
de sélectionner une cible. Aucune série du pli de validation n'entre dans son train.
Pour la semi-supervision inductive, le pool doit être restreint aux groupes du
train concerné, également pendant la validation interne.


In [ ]:
def assign_folds(m: pd.DataFrame, n_outer=5, n_inner=3) -> pd.DataFrame:
    """Plis externes par groupe (GroupKFold déterministe, sans shuffle), communs à toutes
    les cibles ; plis internes par groupe à l'intérieur de chaque entraînement externe."""
    from sklearn.model_selection import GroupKFold
    d = m.loc[~m.is_duplicate_excess, ['raw_row_id', 'group']].reset_index(drop=True)
    d['outer_fold'] = -1
    for k, (_, te) in enumerate(GroupKFold(n_splits=n_outer).split(d, groups=d.group)):
        d.loc[te, 'outer_fold'] = k
    for k in range(n_outer):
        tr = d.index[d.outer_fold != k]
        col = f'inner_fold_when_outer{k}'
        d[col] = -1
        n_g = d.loc[tr, 'group'].nunique()
        for j, (_, te) in enumerate(GroupKFold(n_splits=min(n_inner, n_g)).split(tr, groups=d.loc[tr, 'group'])):
            d.loc[tr[te], col] = j
    return d


In [ ]:
folds = assign_folds(master)
assert folds.groupby('group').outer_fold.nunique().max() == 1
for k in range(5):
    train_groups = set(folds.loc[folds.outer_fold != k, 'group'])
    valid_groups = set(folds.loc[folds.outer_fold == k, 'group'])
    assert train_groups.isdisjoint(valid_groups)
    inner = folds.loc[folds.outer_fold != k]
    assert inner.groupby('group')[f'inner_fold_when_outer{k}'].nunique().max() == 1
d = cohort.merge(folds, on=['raw_row_id', 'group'], validate='one_to_one')
display(d.groupby('outer_fold').agg(
    groupes=('group', 'nunique'), lignes=('raw_row_id', 'size'),
    YS=('has_yield_strength', 'sum'), UTS=('has_uts', 'sum'), A=('has_elongation', 'sum')))


## 5. Preprocessing appris dans chaque pli

Moyennes/écarts-types et catégories du train, puis imputation itérative
BayesianRidge (initialisation médiane, max_iter=100, tol=1e-3, ordre ascendant).
Les imputations non négatives sont bornées ; les valeurs observées restent intactes.

Les couples PWHT imputés viennent du train. Une composante observée impose un
couple compatible : **aucun couple compatible → erreur explicite**, plutôt qu'une
paire inventée. Une colonne entièrement absente ou une non-convergence est aussi
refusée. La médiane reste disponible uniquement comme contrôle explicite.


In [ ]:
class FoldPreprocessor(BaseEstimator, TransformerMixin):
    """À ajuster sur les lignes d'entraînement d'un pli uniquement.

    1. standardisation des entrées numériques (moyenne/écart-type du train, NaN ignorés) ;
    2. one-hot du procédé (catégories du train ; inconnue -> vecteur nul) ;
    3. imputation : 'iterative' (IterativeImputer/BayesianRidge, init médiane, ordre ascendant,
       max_iter=100, tol=1e-3, random_state=42, bornes physiques >= 0) ou 'median' ;
    4. PWHT : un couple (température, durée) imputé est projeté sur le couple réel du train
       le plus proche (distance en écarts-types du train), en respectant une valeur observée.
    Une colonne entièrement absente du train est refusée plutôt que fabriquée.
    Sortie : entrées standardisées + indicatrices du procédé. Les cibles n'entrent jamais ici.
    """

    def __init__(self, num_cols, cat_col=PROCESS_COL, method='iterative', max_iter=100,
                 tol=1e-3, random_state=SEED):
        self.num_cols, self.cat_col, self.method = num_cols, cat_col, method
        self.max_iter, self.tol, self.random_state = max_iter, tol, random_state

    def _encode(self, X):
        Z = (X[self.num_cols].to_numpy(float) - self.mu_) / self.sd_
        C = np.stack([(X[self.cat_col] == c).to_numpy(float) for c in self.cats_], axis=1) \
            if self.cat_col else np.empty((len(X), 0))
        return np.hstack([Z, C])

    def fit(self, X, y=None):
        Xn = X[self.num_cols].astype(float)
        empty = Xn.columns[Xn.notna().sum() == 0].tolist()
        if empty:
            raise ValueError(f'Colonnes entièrement absentes du train : {empty}')
        self.mu_ = Xn.mean().to_numpy()
        self.sd_ = Xn.std(ddof=0).replace(0, 1).to_numpy()
        self.cats_ = sorted(X[self.cat_col].dropna().unique()) if self.cat_col else []
        Z = self._encode(X)
        lo = np.full(Z.shape[1], -np.inf)
        for i, c in enumerate(self.num_cols):
            if c in NONNEG:
                lo[i] = (0 - self.mu_[i]) / self.sd_[i]
        if self.method == 'iterative':
            self.imp_ = IterativeImputer(estimator=BayesianRidge(), initial_strategy='median',
                                         imputation_order='ascending', max_iter=self.max_iter,
                                         tol=self.tol, random_state=self.random_state,
                                         skip_complete=False, min_value=lo, keep_empty_features=True)
        elif self.method == 'median':
            self.imp_ = SimpleImputer(strategy='median', keep_empty_features=True)
        else:
            raise ValueError(self.method)
        with warnings.catch_warnings():
            warnings.simplefilter('ignore', ConvergenceWarning)
            self.imp_.fit(Z)
        self.converged_ = (self.method != 'iterative') or (self.imp_.n_iter_ < self.max_iter)
        self.n_iter_ = getattr(self.imp_, 'n_iter_', 0)
        if not self.converged_:
            raise RuntimeError('Imputation itérative non convergée : réviser le schéma dans le train.')
        # couples PWHT réels du train
        self.pw_idx_ = [self.num_cols.index(c) for c in ('pwht_temp_C', 'pwht_time_h')
                        if c in self.num_cols]
        if len(self.pw_idx_) == 2:
            P = Z[:, self.pw_idx_]
            self.pw_pairs_ = np.unique(P[~np.isnan(P).any(axis=1)], axis=0)
            if len(self.pw_pairs_) == 0:
                raise ValueError('Aucun couple PWHT complet dans le train.')
        return self

    def transform(self, X):
        Z = self._encode(X)
        miss = np.isnan(Z)
        out = self.imp_.transform(Z)
        if len(self.pw_idx_) == 2 and len(self.pw_pairs_):
            i, j = self.pw_idx_
            for r in np.where(miss[:, i] | miss[:, j])[0]:
                cand = self.pw_pairs_
                if not miss[r, i]:   # température observée : couples compatibles seulement
                    cand = cand[np.isclose(cand[:, 0], Z[r, i])]
                if not miss[r, j]:
                    cand = cand[np.isclose(cand[:, 1], Z[r, j])]
                if len(cand) == 0:
                    raise ValueError(f'Aucun couple PWHT compatible pour la ligne {X.index[r]}.')
                best = cand[np.argmin(((cand - out[r, [i, j]]) ** 2).sum(axis=1))]
                out[r, i] = Z[r, i] if not miss[r, i] else best[0]
                out[r, j] = Z[r, j] if not miss[r, j] else best[1]
        return out

    def get_feature_names_out(self, input_features=None):
        return np.array([str(c) for c in self.num_cols] + [f'{self.cat_col}={c}' for c in self.cats_], dtype=object)


In [ ]:
sc = task_schema('yield_strength')
tr = d.loc[(d.outer_fold != 0) & d.has_yield_strength]
te = d.loc[(d.outer_fold == 0) & d.has_yield_strength]
pre = FoldPreprocessor(sc['num']).fit(tr)
Xte = pre.transform(te)
print('Convergence :', pre.converged_, '| itérations :', pre.n_iter_)
print('Features :', list(pre.get_feature_names_out()))
restored = Xte[:, :len(sc['num'])] * pre.sd_ + pre.mu_
original = te[sc['num']].to_numpy(float)
observed = np.isfinite(original)
assert np.isfinite(Xte).all()
assert np.allclose(restored[observed], original[observed])


## 6. Contrôle de sensibilité médiane / itérative

Reproduction du benchmark AI audité : mêmes plis et lignes, modèles à paramètres
fixes. Ce contrôle concerne le choix d'imputation, **pas une sélection finale des
modèles**. Les plis internes seront utilisés pour régler les modèles à l'étape 03.
Les résultats de reconstruction de l'ancienne EDA sur 11 entrées restent dans le
rapport historique ; ils ne constituent pas une nouvelle mesure sur cette cohorte 13.


In [ ]:
from sklearn.linear_model import Ridge
from sklearn.ensemble import RandomForestRegressor
from sklearn.dummy import DummyRegressor
from sklearn.metrics import mean_absolute_error, mean_squared_error, r2_score
MODELS = {'baseline_median': lambda: DummyRegressor(strategy='median'), 'ridge': lambda: Ridge(alpha=1.0),
          'rf': lambda: RandomForestRegressor(n_estimators=300, min_samples_leaf=2, random_state=SEED, n_jobs=-1)}
rows = []
for task in TENSILE:
    sc = task_schema(task); dt = d[d[task].notna()]
    for k in range(5):
        tr, te = dt[dt.outer_fold != k], dt[dt.outer_fold == k]
        for imp in ['median', 'iterative']:
            pre = FoldPreprocessor(sc['num'], method=imp).fit(tr)
            Xtr, Xte = pre.transform(tr), pre.transform(te)
            for name, mk in MODELS.items():
                if name == 'baseline_median' and imp == 'iterative': continue
                p = mk().fit(Xtr, tr[task]).predict(Xte); y = te[task]
                rows.append(dict(task=task, fold=k, model=name, imputer='-' if name == 'baseline_median' else imp,
                                 n_train=len(tr), n_test=len(te), MAE=mean_absolute_error(y, p),
                                 RMSE=mean_squared_error(y, p) ** .5, R2=r2_score(y, p),
                                 converged=pre.converged_, n_iter=pre.n_iter_))
bench = pd.DataFrame(rows)
print('Ajustements itératifs convergés :', bench.query("imputer == 'iterative'").drop_duplicates(['task', 'fold']).converged.all())
display(bench.groupby(['task', 'model', 'imputer'])[['MAE', 'RMSE', 'R2']].agg(['mean', 'std']).round(2))


In [ ]:
paired = bench.loc[bench.model != 'baseline_median'].pivot_table(
    index=['task', 'model', 'fold'], columns='imputer', values='MAE')
gain = (paired['median'] - paired['iterative']).rename('gain_MAE_iterative')
display(gain.groupby(['task', 'model']).agg(['mean', 'std', 'min', 'max']))
assert bench.loc[bench.imputer == 'iterative', 'converged'].all()


L'itérative converge sur les trois tâches et améliore plusieurs scores, notamment
UTS et allongement. Elle n'est pas supérieure sur chaque tâche/pli : la médiane
est légèrement devant sur YS avec Random Forest. Les R² restent modestes ; leurs
causes ne sont pas isolées. La méthode itérative choisie est conservée, la médiane
sert de contrôle. Les écarts-types entre plis ne sont pas des tests de significativité.


## 7. Exports avant apprentissage

`data/processed/modeling/` contient la table maître, les statuts par cellule, les
alias, les groupes, les plis, la quarantaine et les signatures Charpy. Le manifeste
enregistre les hashes, les unités, versions, paramètres et effectifs.

`master.csv` contient encore les alias **marqués** et les lignes non étiquetées.
Pour le supervisé : `in_strict13 & has_<cible>`. Pour le non étiqueté :
`in_strict13 & ~has_<cible>`, dans les groupes d'entraînement uniquement.
Aucune matrice imputée sur toute la base n'est exportée pour la validation.


In [ ]:
UNITS = {**dict.fromkeys(RAW_COLS[:12], 'wt%'), **dict.fromkeys(RAW_COLS[12:21], 'ppm'),
         **dict.fromkeys(MICRO + ['elongation', 'reduction_area'], '%'),
         'current_A': 'A', 'voltage_V': 'V', 'heat_input_kJmm': 'kJ/mm',
         'interpass_C': 'degC', 'pwht_temp_C': 'degC', 'pwht_time_h': 'h',
         'yield_strength': 'MPa', 'uts': 'MPa', 'charpy_temp_C': 'degC', 'charpy_J': 'J',
         'hardness': 'kg/mm2; Vickers protocol tracked separately', 'fatt50': 'unresolved'}
tables = {'master.csv': master, 'cell_provenance.csv': prov, 'duplicate_aliases.csv': aliases,
          'groups.csv': groups, 'folds.csv': folds, 'unit_review_cells.csv': rev,
          'charpy_protocol_candidates.csv': cp.reset_index(), 'imputer_benchmark.csv': bench}
for filename, table in tables.items():
    table.to_csv(OUT / filename, index=False)
notebook_path = PROJECT_ROOT / 'notebooks/02_data_preparation.ipynb'
notebook = json.loads(notebook_path.read_text())
source = [{'cell_type': cell['cell_type'], 'source': ''.join(cell['source'])}
          for cell in notebook['cells']]
source_hash = hashlib.sha256(json.dumps(source, ensure_ascii=False, sort_keys=True).encode()).hexdigest()
manifest = {
    'protocol_version': 'notebooks-v1', 'raw_sha256': sha256(RAW),
    'notebook_source_sha256': source_hash,
    'created_at_utc': datetime.now(timezone.utc).isoformat(),
    'versions': {'python': platform.python_version(), 'pandas': pd.__version__,
                 'numpy': np.__version__, 'sklearn': sklearn.__version__},
    'seed': SEED, 'feature_sets': FEATURE_SETS, 'primary_set': PRIMARY_SET, 'units': UNITS,
    'process_synonyms': PROCESS_SYNONYMS, 'merge_name_variants': MERGE_NAME_VARIANTS,
    'cohort': {'max_unavailable_primary_numeric_inputs': 2, 'exclude_duplicate_excess': True},
    'imputer': {'method': 'iterative', 'estimator': 'BayesianRidge', 'initial_strategy': 'median',
                'max_iter': 100, 'tol': 1e-3, 'order': 'ascending', 'lower_bound': 0,
                'pwht_pair_projection': True, 'no_compatible_pair': 'error',
                'nonconvergence': 'error'},
    'folds': {'outer': 'GroupKFold(5), no shuffle', 'inner': 'GroupKFold(3) per outer train'},
    'benchmark_models': {'constant': {'strategy': 'median'}, 'ridge': {'alpha': 1.0},
                         'random_forest': {'n_estimators': 300, 'min_samples_leaf': 2,
                                           'random_state': SEED}},
    'counts': {'raw': len(master), 'after_dedup': len(dedup), 'groups': int(dedup.group.nunique()),
               'strict13_rows': len(cohort), 'strict13_groups': int(cohort.group.nunique()),
               'unit_review_cells': len(rev), 'unlabeled_all': int(dedup.is_unlabeled_all.sum())},
    'files': {filename: sha256(OUT / filename) for filename in sorted(tables)},
}
(OUT / 'manifest.json').write_text(json.dumps(manifest, indent=2, ensure_ascii=False) + '\n')
print('Exports :', OUT)
display(pd.Series(manifest['counts']))


## 8. Passage aux modèles

Ouvrir `03_modeling.ipynb` pour examiner les cohortes et préparer la comparaison.
Les modèles finaux, leur réglage et la méthode semi-supervisée restent à réaliser.
Charpy, les unités en quarantaine, la sensibilité aux groupes et aux jeux 11/16
restent des travaux distincts. Les anciens CSV de `ai_prepared/` ne sont pas les
entrées de ce nouveau protocole.
